# Blackbird (MIT) + Spyx New Architectures

This notebook targets the Blackbird UAV dataset.

Status in Tonic:
- No direct Blackbird loader in `tonic.datasets` in this environment.

Goal:
- Use local Blackbird exports if available.
- Compare Spyx architectures under fast motion dynamics.

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/Blackbird")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_blackbird():
    import tonic.datasets as d
    return any("blackbird" in n.lower() for n in dir(d))

print("tonic_has_blackbird =", tonic_has_blackbird())
print("expected local artifacts: imu.csv, trajectory.csv, events.* or frames/")

In [ ]:
def make_blackbird_target(batch=4, dim=6):
    return jnp.zeros((batch, dim), dtype=jnp.float32)


def synthetic_blackbird_batch(batch=4, T=20, H=64, W=64, C=2, seed=1):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.03, size=(T, batch, H, W, C)).astype(np.float32)
    return jnp.asarray(x), make_blackbird_target(batch, dim=6)


def blackbird_model_bank(input_hw=(64, 64), input_channels=2, output_dim=6):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=32, channels2=64, output_dim=output_dim)
    sparse_cfg = fm.SparseConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=32, channels2=64, output_dim=output_dim, event_threshold=0.0)
    return {
        "conv": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "ternary": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
        "sparse": lambda x: fm.SparseEventConvLIFSNN(sparse_cfg)(x),
    }


x, y = synthetic_blackbird_batch()
models = blackbird_model_bank()

for name, fn in models.items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x)
    pred, aux = net.apply(params, x)
    mse = jnp.mean((pred - y) ** 2)
    print(name, "pred", pred.shape, "mse", float(mse), "spike_rate", np.asarray(aux["spike_rate"]))

## Next Steps for Real Blackbird Runs

1. Export synchronized event/image and IMU streams to `../data/Blackbird`.
2. Build a sequence index with train/val/test by flight condition.
3. Extend the notebook with trajectory loss terms and multi-sequence evaluation.